### Data Loading

In [329]:
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import entropy
import matplotlib.pyplot as plt
from tqdm import tqdm
import os

plt.style.use(
    #'../../Dataset-And-Solve-Class/class-12-ML-how-machine-learns/class-12-deeplearning.mplstyle' #my_windows
    '../Dataset-And-Solve-Class/class-12-ML-how-machine-learns/class-12-deeplearning.mplstyle' #mac
)

In [330]:
# Define paths
ROOT_DIR = ".." #mac
#ROOT_DIR = "../.." #my_windows
DATA_DIR = os.path.join(ROOT_DIR, "my-practice/Dataset")
DATASET_PATH = os.path.join(DATA_DIR, "housing.csv")

### Data Exploration and Visualization

In [331]:
dataset = pd.read_csv(DATASET_PATH)
dataset.head()

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished


In [332]:
dataset.columns

Index(['price', 'area', 'bedrooms', 'bathrooms', 'stories', 'mainroad',
       'guestroom', 'basement', 'hotwaterheating', 'airconditioning',
       'parking', 'prefarea', 'furnishingstatus'],
      dtype='object')

In [333]:
numerical_cols = dataset.select_dtypes(include='number').columns
categorical_cols = dataset.select_dtypes(include='object').columns

print(numerical_cols)
print(categorical_cols)

Index(['price', 'area', 'bedrooms', 'bathrooms', 'stories', 'parking'], dtype='object')
Index(['mainroad', 'guestroom', 'basement', 'hotwaterheating',
       'airconditioning', 'prefarea', 'furnishingstatus'],
      dtype='object')


### Data Preprocessing

In [334]:
from sklearn.preprocessing import StandardScaler, LabelEncoder
scaler = StandardScaler()
le = LabelEncoder()

In [335]:
dataset[numerical_cols] = scaler.fit_transform(dataset[numerical_cols])
dataset[numerical_cols].head()

,price,area,bedrooms,bathrooms,stories,parking
0,4.566365,1.046726,1.403419,1.421812,1.378217,1.517692
1,4.004484,1.757010,1.403419,5.405809,2.532024,2.679409
2,4.004484,2.218232,0.047278,1.421812,0.224410,1.517692
3,3.985755,1.083624,1.403419,1.421812,0.224410,2.679409
4,3.554979,1.046726,1.403419,-0.570187,0.224410,1.517692


In [336]:
dataset[categorical_cols] = dataset[categorical_cols].apply(
    le.fit_transform
)

dataset[categorical_cols].head()

,mainroad,guestroom,basement,hotwaterheating,airconditioning,prefarea,furnishingstatus
0,1,0,0,0,1,1,0
1,1,0,0,0,1,0,0
2,1,0,1,0,0,1,1
3,1,0,1,0,1,1,0
4,1,1,1,0,1,0,0


### Feature Engineering

In [337]:
# Problem: From the housing dataset, we need to predict, the area in prefered or not [1 -> prefered, 0 -> not prefered]
target_var = 'prefarea'
dataset[target_var].value_counts()

prefarea
0    417
1    128
Name: count, dtype: int64

In [338]:
#select input [X], output [y]
X = dataset.drop(columns=target_var)
y = dataset[target_var]

In [339]:
X.head()

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,furnishingstatus
0,4.566365,1.046726,1.403419,1.421812,1.378217,1,0,0,0,1,1.517692,0
1,4.004484,1.757010,1.403419,5.405809,2.532024,1,0,0,0,1,2.679409,0
2,4.004484,2.218232,0.047278,1.421812,0.224410,1,0,1,0,0,1.517692,1
3,3.985755,1.083624,1.403419,1.421812,0.224410,1,0,1,0,1,2.679409,0
4,3.554979,1.046726,1.403419,-0.570187,0.224410,1,1,1,0,1,1.517692,0


In [340]:
y.head()

0    1
1    0
2    1
3    1
4    0
Name: prefarea, dtype: int64

### Data Splitting Fairly and create Balanced dataset

In [341]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [342]:
y_train.value_counts()

prefarea
0    334
1    102
Name: count, dtype: int64

In [343]:
# as the dataset is imbalance, we will balance the dataset using any technique. We will follow oversamlpling SMOTE technique. It will increase lower value
# to reache upper value and create a balance dataset
from imblearn.over_sampling import SMOTE
smonte = SMOTE(random_state=42)
X_train, y_train = smonte.fit_resample(X_train, y_train) 

y_train.value_counts()

/Users/shajib/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


prefarea
0    334
1    334
Name: count, dtype: int64

# Model Selection

##### Decision Tree Classifier

In [344]:
# selecting decission tree classifier model
from sklearn.tree import DecisionTreeClassifier
decision_tree = DecisionTreeClassifier(random_state=42)

# train the model
decision_tree.fit(X_train, y_train)

DecisionTreeClassifier(random_state=42)

In [345]:
# see the prediction after training
y_pred = decision_tree.predict(X_test)
print(y_pred)

[0 0 0 1 0 0 1 0 0 0 0 0 0 0 0 0 1 1 0 0 0 1 0 0 0 1 0 0 0 0 0 1 0 0 0 0 0
 0 0 0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 1 0 1 1 0 1 0 1
 0 0 1 0 1 0 0 1 1 1 1 0 0 1 1 0 0 0 0 1 0 1 0 0 0 1 0 0 0 0 1 0 0 0 1]


In [346]:
# now, we need to see how much accurate our prediction is
# sklearn.metrics - this package provide all the evaluation matrics

from sklearn.metrics import accuracy_score, classification_report
accuracy = accuracy_score(y_test, y_pred) #giving the real test output [y_test] and the our prediction [y_pred] using model
report = classification_report(y_test, y_pred) #same as above

In [347]:
print(accuracy)

0.7247706422018348


In [348]:
print(report)

              precision    recall  f1-score   support

           0       0.83      0.81      0.82        83
           1       0.43      0.46      0.44        26

    accuracy                           0.72       109
   macro avg       0.63      0.63      0.63       109
weighted avg       0.73      0.72      0.73       109



##### K-Nearest Neighbors (KNN)

In [349]:
#training on train data
from sklearn.neighbors import KNeighborsClassifier
neighbors = KNeighborsClassifier()
neighbors.fit(X_train, y_train)

KNeighborsClassifier()

In [350]:
#predict on test input data
y_pred = neighbors.predict(X_test)
print(y_pred)

[1 0 1 0 1 0 1 1 0 0 0 0 0 0 0 0 1 0 0 1 0 1 0 1 1 1 0 0 1 1 0 0 0 0 0 0 1
 0 0 0 1 0 0 0 0 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 0 0 1 1
 0 1 1 0 1 0 1 1 1 1 0 1 0 0 1 1 0 1 0 0 1 0 1 0 1 1 1 1 0 0 1 0 0 1 0]


In [351]:
# evaluation on test output vs predict
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

In [352]:
print(accuracy)

0.6238532110091743


In [353]:
print(report)

              precision    recall  f1-score   support

           0       0.83      0.64      0.72        83
           1       0.33      0.58      0.42        26

    accuracy                           0.62       109
   macro avg       0.58      0.61      0.57       109
weighted avg       0.71      0.62      0.65       109



##### Support Vector Classifier

In [354]:
from sklearn.svm import SVC
svm_model = SVC(probability=True, random_state=42)
svm_model.fit(X_train, y_train)

SVC(probability=True, random_state=42)

In [355]:
y_pred = svm_model.predict(X_test)
print(y_pred)

[0 0 0 1 0 1 1 0 0 0 0 0 0 0 0 0 1 1 0 0 0 1 0 1 1 1 0 0 0 1 0 0 0 0 0 0 1
 0 0 0 1 0 0 1 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 0 1 0 1 0 1
 0 0 1 0 1 0 0 1 0 1 0 1 0 1 1 0 0 1 0 0 0 0 0 0 1 1 0 0 1 0 1 0 0 1 0]


In [356]:
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print(accuracy)

0.6697247706422018


In [357]:
print(report)

              precision    recall  f1-score   support

           0       0.81      0.73      0.77        83
           1       0.35      0.46      0.40        26

    accuracy                           0.67       109
   macro avg       0.58      0.60      0.59       109
weighted avg       0.70      0.67      0.68       109



# Ensemble Learning

##### Boosting

In [358]:
#select model - DecisionTreeClassifier and boosting model - AdaBoostClassifier
from sklearn.ensemble import AdaBoostClassifier
boosting_model = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(), #model
    n_estimators=3, # 3 times iteration [train-predict and right/wrong output > train-predict on wrong output > train-predict on wrong output...........n times]
    random_state=42
)

#train the model
boosting_model.fit(X_train, y_train)

AdaBoostClassifier(estimator=DecisionTreeClassifier(), n_estimators=3,
                   random_state=42)

In [359]:
# prediction on test
y_pred = boosting_model.predict(X_test)
print(y_pred)

[0 0 0 1 0 1 1 0 0 0 0 0 0 0 0 0 1 1 0 0 0 1 0 0 0 1 0 0 0 0 0 1 0 0 0 1 0
 0 0 0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 1 0 1 1 0 1 1 1
 0 0 1 0 1 0 1 1 1 1 1 0 0 1 0 0 0 0 0 1 0 1 0 0 0 1 0 1 0 0 1 0 0 0 1]


In [360]:
#evaluation
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print(accuracy)
print(report)

0.6880733944954128
              precision    recall  f1-score   support

           0       0.82      0.76      0.79        83
           1       0.38      0.46      0.41        26

    accuracy                           0.69       109
   macro avg       0.60      0.61      0.60       109
weighted avg       0.71      0.69      0.70       109



### Bagging

In [361]:
from sklearn.ensemble import BaggingClassifier
bagging_model = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=10,
    random_state=42
)
bagging_model.fit(X_train, y_train)

BaggingClassifier(estimator=DecisionTreeClassifier(), random_state=42)

In [362]:
y_pred = bagging_model.predict(X_test)
print(y_pred)

[0 1 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 1 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 1 0 1 1 0 1 0 0
 0 0 1 0 0 0 0 0 1 1 0 1 0 1 1 0 0 1 0 0 0 0 0 0 0 1 0 0 0 0 1 0 0 0 1]


In [363]:
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print(accuracy)
print(report)

0.7706422018348624
              precision    recall  f1-score   support

           0       0.83      0.88      0.85        83
           1       0.52      0.42      0.47        26

    accuracy                           0.77       109
   macro avg       0.68      0.65      0.66       109
weighted avg       0.76      0.77      0.76       109



### Stacking

In [364]:
from sklearn.ensemble import StackingClassifier, RandomForestClassifier
# base estimator list
estimators = [
    ('decision_tree', DecisionTreeClassifier(random_state=42)),
    ('knn', KNeighborsClassifier()),
    ('svm', SVC(probability=True, random_state=42))
]
# Initialize and train the Stacking Classifier
stacking_model = StackingClassifier(
    estimators=estimators, 
    final_estimator=RandomForestClassifier(random_state=42) #final estimator
)
stacking_model.fit(X_train, y_train)

StackingClassifier(estimators=[('decision_tree',
                                DecisionTreeClassifier(random_state=42)),
                               ('knn', KNeighborsClassifier()),
                               ('svm', SVC(probability=True, random_state=42))],
                   final_estimator=RandomForestClassifier(random_state=42))

In [365]:
y_pred = stacking_model.predict(X_test)
print(y_pred)

[1 0 0 1 0 0 1 1 0 1 0 0 0 0 0 0 0 1 0 1 0 1 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0
 0 0 1 1 0 0 0 0 1 0 1 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 1 1 0 1 0 1
 0 0 1 0 1 0 0 1 0 1 0 0 0 1 1 1 0 1 0 0 0 0 0 0 0 1 1 0 0 0 1 1 0 0 0]


In [366]:
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print(accuracy)
print(report)

0.6513761467889908
              precision    recall  f1-score   support

           0       0.78      0.75      0.77        83
           1       0.30      0.35      0.32        26

    accuracy                           0.65       109
   macro avg       0.54      0.55      0.54       109
weighted avg       0.67      0.65      0.66       109



### Votting

In [367]:
from sklearn.ensemble import VotingClassifier
# base estimator list
estimators = [
    ('decision_tree', DecisionTreeClassifier(random_state=42)),
    ('knn', KNeighborsClassifier()),
    ('svm', SVC(probability=True, random_state=42))
]
# Initialize and train the Stacking Classifier
votting_model = VotingClassifier(
    estimators=estimators, 
    voting='soft'
)
votting_model.fit(X_train, y_train)

VotingClassifier(estimators=[('decision_tree',
                              DecisionTreeClassifier(random_state=42)),
                             ('knn', KNeighborsClassifier()),
                             ('svm', SVC(probability=True, random_state=42))],
                 voting='soft')

In [368]:
y_pred = stacking_model.predict(X_test)
print(y_pred)

[1 0 0 1 0 0 1 1 0 1 0 0 0 0 0 0 0 1 0 1 0 1 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0
 0 0 1 1 0 0 0 0 1 0 1 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 1 1 0 1 0 1
 0 0 1 0 1 0 0 1 0 1 0 0 0 1 1 1 0 1 0 0 0 0 0 0 0 1 1 0 0 0 1 1 0 0 0]


In [369]:
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print(accuracy)
print(report)

0.6513761467889908
              precision    recall  f1-score   support

           0       0.78      0.75      0.77        83
           1       0.30      0.35      0.32        26

    accuracy                           0.65       109
   macro avg       0.54      0.55      0.54       109
weighted avg       0.67      0.65      0.66       109

